<a href="https://colab.research.google.com/github/mdanmek/nida-dads-notes/blob/main/dads5001-data-tools/project/eda/03_construction_eda_2569.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DADS5001 Mini Project
## การวิเคราะห์เชิงสำรวจโครงการจ้างก่อสร้างภาครัฐ ปีงบประมาณ 2569

Notebook นี้รับข้อมูล 180,079 แถวจาก
`02_construction_data_preparation_2569.ipynb` และจัดให้อยู่ในระดับ
178,978 โครงการก่อนวิเคราะห์ โดยใช้ข้อมูลทุกแถวเฉพาะเมื่อต้องตรวจ
สัญญา ผู้รับจ้าง หรือกิจการร่วมค้า

### คำถามกลาง

> ในโครงการจ้างก่อสร้างเกือบ 180,000 โครงการ
> มีรูปแบบใดที่ช่วยคัดกรองโครงการซึ่งควรเปิดเอกสารตรวจสอบก่อน?

### คำถามย่อย

1. โครงการมีขนาดและมูลค่ากระจายตัวอย่างไร
2. รูปแบบใกล้ 500,000 บาทสัมพันธ์กับวิธีจัดซื้อหรือไม่
3. ราคาที่ตกลงสัมพันธ์กับงบประมาณและราคากลางอย่างไร
4. จำนวนและมูลค่ากระจายต่างกันตามหน่วยงานและพื้นที่หรือไม่
5. มูลค่ากระจุกตัวอยู่ในผู้รับจ้างบางส่วนมากน้อยเพียงใด
6. ข้อค้นพบใดควรพัฒนาเป็นตัวชี้วัดใน Notebook ถัดไป

### ขอบเขตและหลักการตีความ

- ใช้ข้อมูลเฉพาะ `จ้างก่อสร้าง` ปีงบประมาณ 2569
- ใช้ `รหัสโครงการ` เป็นหน่วยหลักของ EDA
- ใช้ข้อมูลทุกแถวเมื่อวิเคราะห์ผู้รับจ้างและรายละเอียดสัญญา
- เก็บกิจการร่วมค้าเป็น Supplier Entity และไม่นับมูลค่าแถวสมาชิกซ้ำ
- ใช้ Pandas, NumPy, Matplotlib และ Seaborn
- ยังไม่กำหนดลำดับตรวจสอบจนกว่าจะเห็นข้อค้นพบครบทุกมิติ

> รูปแบบที่ผิดสังเกตเป็นเบาะแสสำหรับตั้งคำถาม
> ไม่ใช่หลักฐานยืนยันการทุจริต


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option(
    'display.float_format',
    lambda value: f'{value:,.2f}'
)

sns.set_theme(style='whitegrid')

data_path = Path(
    '/content/drive/MyDrive/learning/dads/dads5001/'
    'project_1_dads5001/dataset/procurement/egp-contract/'
    'processed/construction_contracts_2569.csv'
)

project_directory = data_path.parents[4]
figure_directory = (
    project_directory
    / 'dads5001-data-tools'
    / 'project'
    / 'eda'
    / 'figure'
)
figure_directory.mkdir(parents=True, exist_ok=True)

construction_data = pd.read_csv(
    data_path,
    low_memory=False
)

print(f'Shape: {construction_data.shape}')
print(f'Figure directory: {figure_directory}')

## 1. ทำความเข้าใจข้อมูล

เริ่มจากตรวจสอบโครงสร้าง ชนิดข้อมูล ค่าว่าง และจำนวนค่าที่ไม่ซ้ำ
ของแต่ละคอลัมน์ เพื่อเลือกตัวแปรที่สามารถนำมาใช้วิเคราะห์ได้จริง

การตรวจในส่วนนี้มุ่งเฉพาะข้อมูลที่จำเป็นต่อ EDA
ไม่ใช่การตรวจสอบคุณภาพข้อมูลอย่างละเอียด

In [ ]:
display(construction_data.head())

In [ ]:
field_summary = pd.DataFrame({
    'column': construction_data.columns,
    'dtype': (
        construction_data
        .dtypes
        .astype(str)
        .values
    ),
    'non_null': (
        construction_data
        .notna()
        .sum()
        .values
    ),
    'missing': (
        construction_data
        .isna()
        .sum()
        .values
    ),
    'missing_pct': (
        construction_data
        .isna()
        .mean()
        .mul(100)
        .values
    ),
    'unique': (
        construction_data
        .nunique(
            dropna=True
        )
        .values
    )
})

display(field_summary)

### สรุปการทำความเข้าใจข้อมูล

ข้อมูลจ้างก่อสร้างมี 180,079 แถว และ 29 คอลัมน์
ตัวแปรหลักด้านงบประมาณ ราคา วิธีจัดซื้อ หน่วยงาน และพื้นที่
มีข้อมูลค่อนข้างครบถ้วน

คอลัมน์ `ราคากลาง (บาท)` ขาดข้อมูล 107 แถว หรือ 0.06%
และ `วงเงินงบประมาณในสัญญา (บาท)` ขาดข้อมูล 90 แถว
หรือ 0.05% ส่วนข้อมูลพิกัดขาด 1,438 แถว หรือ 0.80%

คอลัมน์ต่อไปนี้มีเพียงค่าเดียว จึงไม่ช่วยในการเปรียบเทียบ
ภายในชุดข้อมูลจ้างก่อสร้าง:

- ชื่อประเภทโครงการ
- ชื่อกลุ่มวิธีการจัดซื้อจัดจ้าง
- ปีงบประมาณ
- สถานะโครงการ
- สถานะสัญญา

ก่อนสร้างข้อมูลระดับโครงการ จะตรวจสอบว่าแถวที่ใช้
`รหัสโครงการ` เดียวกันมีค่าของตัวแปรระดับโครงการ
สอดคล้องกันหรือไม่

In [ ]:
project_columns = [
    'ชื่อโครงการจัดซื้อจัดจ้าง',
    'ชื่อหน่วยงาน',
    'ชื่อหน่วยงานย่อย',
    'ชื่อวิธีการจัดซื้อจัดจ้าง',
    'วันที่ประกาศจัดซื้อจัดจ้าง',
    'วงเงินงบประมาณ (บาท)',
    'ราคากลาง (บาท)',
    (
        'ราคาที่ตกลงซื้อ / จ้าง '
        'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
    ),
    'วันที่เกิดรายการ',
    'จังหวัด',
    'เขต/อำเภอ',
    'แขวง/ตำบล'
]

project_consistency = (
    construction_data
    .groupby('รหัสโครงการ')[
        project_columns
    ]
    .nunique(
        dropna=False
    )
)

inconsistent_projects = (
    project_consistency
    .gt(1)
    .sum()
    .rename(
        'projects_with_multiple_values'
    )
    .to_frame()
)

display(inconsistent_projects)

## 2. เตรียมข้อมูลระดับโครงการ

จากการตรวจสอบโครงการที่ปรากฏมากกว่าหนึ่งแถว พบว่าตัวแปร
ระดับโครงการทั้งหมดมีค่าตรงกันภายใน `รหัสโครงการ` เดียวกัน
ได้แก่ ชื่อโครงการ หน่วยงาน วิธีจัดซื้อ วันที่ ราคา และพื้นที่

ดังนั้น สามารถสร้างชุดข้อมูลระดับโครงการโดยเก็บหนึ่งแถวต่อ
`รหัสโครงการ` ได้ โดยไม่ทำให้ข้อมูลระดับโครงการสูญหาย

ชุดข้อมูลที่ใช้ต่อจากนี้แบ่งเป็น:

- `project_data` สำหรับวิเคราะห์จำนวนโครงการ ราคา หน่วยงาน
  วิธีจัดซื้อ และพื้นที่
- `construction_data` สำหรับวิเคราะห์ผู้ชนะและรายละเอียดสัญญา

In [ ]:
project_data = (
    construction_data
    .drop_duplicates(
        subset='รหัสโครงการ',
        keep='first'
    )
    .copy()
)

print(
    f'Contract-level rows: '
    f'{len(construction_data):,}'
)

print(
    f'Project-level rows: '
    f'{len(project_data):,}'
)

print(
    f'Rows removed from project-level analysis: '
    f'{len(construction_data) - len(project_data):,}'
)

In [ ]:
money_columns = [
    'วงเงินงบประมาณ (บาท)',
    'ราคากลาง (บาท)',
    (
        'ราคาที่ตกลงซื้อ / จ้าง '
        'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
    )
]

money_summary = (
    project_data[
        money_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

display(money_summary)

non_positive_summary = pd.DataFrame({
    'column': money_columns,
    'zero_or_negative': [
        project_data[column]
        .le(0)
        .sum()
        for column in money_columns
    ],
    'missing': [
        project_data[column]
        .isna()
        .sum()
        for column in money_columns
    ]
})

display(non_positive_summary)

### สรุปข้อมูลตัวแปรด้านการเงิน

ชุดข้อมูลระดับโครงการมีจำนวน 178,978 โครงการ
และไม่มีวงเงินงบประมาณหรือราคาที่ตกลงเป็นศูนย์หรือติดลบ

วงเงินงบประมาณมีค่ามัธยฐาน 395,000 บาท ขณะที่ค่าเฉลี่ย
อยู่ที่ประมาณ 2.66 ล้านบาท และมีค่าสูงสุดมากกว่า 4 พันล้านบาท
แสดงว่าการกระจายของมูลค่าโครงการมีลักษณะเบ้ขวาอย่างมาก
จากโครงการมูลค่าสูงจำนวนไม่มาก

นอกจากนี้ 75% ของโครงการมีวงเงินงบประมาณไม่เกิน 500,000 บาท
จึงควรสำรวจว่ามีการกระจุกตัวบริเวณระดับมูลค่าบางช่วงหรือไม่

คอลัมน์ราคากลางขาดข้อมูล 105 โครงการ คิดเป็นสัดส่วนต่ำมาก
จึงไม่ตัดโครงการเหล่านี้ออกจากชุดข้อมูลหลัก แต่จะไม่นำมาใช้
เฉพาะในการวิเคราะห์ที่ต้องเปรียบเทียบกับราคากลาง

In [ ]:
budget_column = 'วงเงินงบประมาณ (บาท)'
reference_price_column = 'ราคากลาง (บาท)'

awarded_price_column = (
    'ราคาที่ตกลงซื้อ / จ้าง '
    'ซึ่งรวมทุกสัญญาในโครงการ (บาท)'
)

# Positive value means the awarded price is below the budget
project_data['budget_saving'] = (
    project_data[budget_column]
    - project_data[awarded_price_column]
)

project_data['budget_saving_pct'] = (
    project_data['budget_saving']
    .div(project_data[budget_column])
    .mul(100)
)

# Positive value means the awarded price is below the reference price
project_data['reference_discount'] = (
    project_data[reference_price_column]
    - project_data[awarded_price_column]
)

project_data['reference_discount_pct'] = (
    project_data['reference_discount']
    .div(
        project_data[
            reference_price_column
        ]
    )
    .mul(100)
)

In [ ]:
price_difference_columns = [
    'budget_saving',
    'budget_saving_pct',
    'reference_discount',
    'reference_discount_pct'
]

display(
    project_data[
        price_difference_columns
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

price_relationship_summary = pd.DataFrame({
    'comparison': [
        'Awarded price below budget',
        'Awarded price equal to budget',
        'Awarded price above budget',
        'Awarded price below reference price',
        'Awarded price equal to reference price',
        'Awarded price above reference price'
    ],
    'project_count': [
        project_data['budget_saving'].gt(0).sum(),
        project_data['budget_saving'].eq(0).sum(),
        project_data['budget_saving'].lt(0).sum(),
        project_data['reference_discount'].gt(0).sum(),
        project_data['reference_discount'].eq(0).sum(),
        project_data['reference_discount'].lt(0).sum()
    ]
})

price_relationship_summary['project_pct'] = (
    price_relationship_summary['project_count']
    .div(len(project_data))
    .mul(100)
)

display(price_relationship_summary)

### ข้อค้นพบจากส่วนต่างราคา

โครงการจำนวน 67.42% มีราคาตกลงต่ำกว่าวงเงินงบประมาณ
ขณะที่ 32.41% มีราคาตกลงเท่ากับวงเงินงบประมาณ และ 0.18%
มีราคาตกลงสูงกว่าวงเงินงบประมาณ

เมื่อเปรียบเทียบกับราคากลาง พบว่า 77.81% ของโครงการ
มีราคาตกลงต่ำกว่าราคากลาง 21.69% มีราคาเท่ากัน และ 0.44%
มีราคาตกลงสูงกว่าราคากลาง

อย่างไรก็ตาม การกระจายของส่วนต่างราคาร้อยละมีค่าผิดสังเกตรุนแรง
โดยเฉพาะ `reference_discount_pct` ซึ่งมีค่าต่ำสุดถึง
-10,399,900% ค่าดังกล่าวอาจเกิดจากราคากลางที่ต่ำผิดปกติ
จนทำให้การหารด้วยราคากลางสร้างค่าร้อยละที่สูงมาก

ดังนั้น การวิเคราะห์จะให้ความสำคัญกับค่ามัธยฐานและ percentile
มากกว่าค่าเฉลี่ย และจะแยกตรวจสอบกรณีราคาตกลงสูงกว่ากรอบราคา
ใน Notebook ตัวชี้วัดเพื่อการตรวจสอบ

## 3. การวิเคราะห์เชิงสำรวจ

### 3.1 โครงการส่วนใหญ่มีขนาดเล็กเพียงใด

ค่าเฉลี่ยสูงกว่าค่ามัธยฐานมาก บ่งชี้ว่ามีโครงการมูลค่าสูงจำนวนน้อย
ดึงค่าเฉลี่ยขึ้น ส่วนโครงการส่วนใหญ่อยู่ในช่วงมูลค่าต่ำกว่า

กราฟแรกแสดงข้อมูลถึง percentile ที่ 99 หมายถึงขอบเขตที่ครอบคลุม
99% ของโครงการ เพื่อให้เห็นกลุ่มหลักชัดขึ้นโดยไม่ได้ลบโครงการมูลค่าสูง
ออกจากการคำนวณ กราฟที่สองใช้ log scale ซึ่งย่อช่วงตัวเลขที่ห่างกันมาก
เพื่อให้เห็นทั้งโครงการขนาดเล็กและขนาดใหญ่ในภาพเดียว

หลังเห็นการกระจายแล้ว จะตรวจช่วงงบประมาณเพื่อค้นหาจุดที่ข้อมูล
กระจุกตัวเป็นพิเศษ


In [ ]:
project_data['budget_million'] = (
    project_data[budget_column]
    .div(1_000_000)
)

project_data['log10_budget'] = np.log10(
    project_data[budget_column]
)

budget_p99 = (
    project_data['budget_million']
    .quantile(0.99)
)

print(
    f'99th percentile of budget: '
    f'{budget_p99:,.2f} million THB'
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

sns.histplot(
    data=project_data,
    x='budget_million',
    bins=50,
    ax=axes[0],
    color='#4C78A8'
)

axes[0].set_xlim(
    0,
    budget_p99
)

axes[0].set_title(
    'Project Budget Distribution '
    '(Up to 99th Percentile)'
)

axes[0].set_xlabel(
    'Budget (million THB)'
)

axes[0].set_ylabel(
    'Number of projects'
)

sns.histplot(
    data=project_data,
    x='log10_budget',
    bins=50,
    ax=axes[1],
    color='#E67E22'
)

axes[1].set_title(
    'Project Budget Distribution '
    '(Log Scale)'
)

axes[1].set_xlabel(
    'Log10 of budget (THB)'
)

axes[1].set_ylabel(
    'Number of projects'
)

plt.tight_layout()

figure_path = figure_directory / 'fig03_01_project_budget_distribution.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

In [ ]:
common_budget_values = (
    project_data[budget_column]
    .value_counts()
    .head(15)
    .rename_axis('budget')
    .reset_index(name='project_count')
)

common_budget_values['budget_million'] = (
    common_budget_values['budget']
    .div(1_000_000)
)

common_budget_values['project_pct'] = (
    common_budget_values['project_count']
    .div(len(project_data))
    .mul(100)
)

display(
    common_budget_values[
        [
            'budget',
            'budget_million',
            'project_count',
            'project_pct'
        ]
    ]
)

### ข้อค้นพบจากการกระจายของวงเงินงบประมาณ

วงเงินงบประมาณของโครงการจ้างก่อสร้างมีการกระจายแบบเบ้ขวา
อย่างชัดเจน โดย 76.34% ของโครงการมีวงเงินไม่เกิน 500,000 บาท
ขณะที่ 99% มีวงเงินไม่เกิน 32.50 ล้านบาท แต่ยังมีโครงการ
มูลค่าสูงมากบางรายการซึ่งทำให้ค่าเฉลี่ยสูงกว่าค่ามัธยฐานมาก

นอกจากนี้ พบว่าวงเงินที่ปรากฏบ่อยที่สุดคือ 500,000 บาท
จำนวน 6,523 โครงการ และพบโครงการจำนวนมากที่มีวงเงิน
ใกล้เคียงแต่ต่ำกว่า 500,000 บาท เช่น 499,000, 498,000,
497,000, 496,000, 495,000 และ 490,000 บาท

รูปแบบดังกล่าวแสดงถึงการกระจุกตัวของวงเงินบริเวณ 500,000 บาท
แต่ยังไม่สามารถอธิบายสาเหตุได้จากข้อมูลชุดนี้เพียงอย่างเดียว
จึงควรนำไปเปรียบเทียบกับวิธีจัดซื้อ หน่วยงาน และลักษณะโครงการ
ในขั้นต่อไป

In [ ]:
budget_bins = [
    0,
    100_000,
    200_000,
    300_000,
    400_000,
    500_000,
    1_000_000,
    5_000_000,
    10_000_000,
    50_000_000,
    np.inf
]

budget_labels = [
    '≤100K',
    '100K–200K',
    '200K–300K',
    '300K–400K',
    '400K–500K',
    '500K–1M',
    '1M–5M',
    '5M–10M',
    '10M–50M',
    '>50M'
]

project_data['budget_band'] = pd.cut(
    project_data[budget_column],
    bins=budget_bins,
    labels=budget_labels,
    include_lowest=True,
    right=True
)

budget_band_summary = (
    project_data['budget_band']
    .value_counts(sort=False)
    .rename('project_count')
    .reset_index()
)

budget_band_summary['project_pct'] = (
    budget_band_summary['project_count']
    .div(len(project_data))
    .mul(100)
)

display(budget_band_summary)

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 6)
)

bars = ax.barh(
    budget_band_summary['budget_band'],
    budget_band_summary['project_pct'],
    color='#4C78A8'
)

ax.bar_label(
    bars,
    labels=[
        f'{value:.1f}%'
        for value
        in budget_band_summary['project_pct']
    ],
    padding=3
)

ax.invert_yaxis()

ax.set_title(
    'Distribution of Construction Projects '
    'by Budget Band'
)

ax.set_xlabel('Share of projects (%)')
ax.set_ylabel('Budget band (THB)')

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_02_construction_projects_by_budget_band.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

### ข้อค้นพบ: จุดกระจุกตัวใกล้ 500,000 บาท

**พบอะไร:** 76.34% ของโครงการมีงบไม่เกิน 500,000 บาท
และช่วง 400,000–500,000 บาทมีสัดส่วนสูงที่สุด 25.09%
ขณะที่โครงการเกิน 50 ล้านบาทมีเพียง 0.49%

**หมายความว่าอย่างไร:** จำนวนโครงการสะท้อนกิจกรรมขนาดเล็กเป็นหลัก
จึงต้องพิจารณาจำนวนและมูลค่ารวมควบคู่กัน ไม่เช่นนั้นโครงการขนาดเล็ก
จะกำหนดภาพรวมมากเกินไป

**คำถามถัดไป:** การกระจุกตัวใกล้ 500,000 บาทสัมพันธ์กับวิธีจัดซื้อ
ประเภทใดเป็นพิเศษหรือไม่ การกระจายเพียงอย่างเดียวยังไม่เพียงพอ
สำหรับสรุปสาเหตุหรือความผิดปกติ


### 3.2 วงเงินงบประมาณ ราคากลาง และราคาที่ตกลง

ส่วนนี้เปรียบเทียบราคาที่ตกลงกับวงเงินงบประมาณและราคากลาง
เพื่อดูว่าค่าทั้งสามเคลื่อนไหวสอดคล้องกันเพียงใด และมีโครงการ
ที่อยู่เหนือหรือต่ำกว่าเส้นราคาเท่ากันในลักษณะใด

เนื่องจากข้อมูลมีจำนวนมากและมูลค่ามีช่วงกว้าง จึงสุ่มข้อมูล
10,000 โครงการสำหรับการแสดง scatter plot และใช้ log scale
เพื่อให้มองเห็นทั้งโครงการขนาดเล็กและขนาดใหญ่ได้ในภาพเดียว

การสุ่มใช้ `random_state=42` เพื่อให้ได้ตัวอย่างเดิมเมื่อรันซ้ำ
ส่วนการคำนวณสถิติยังคงใช้ข้อมูลทุกโครงการ

In [ ]:
price_plot_data = (
    project_data
    .loc[
        project_data[
            reference_price_column
        ].notna()
    ]
    .sample(
        n=min(
            10_000,
            project_data[
                reference_price_column
            ].notna().sum()
        ),
        random_state=42
    )
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)

comparisons = [
    (
        budget_column,
        'Awarded Price vs Budget'
    ),
    (
        reference_price_column,
        'Awarded Price vs Reference Price'
    )
]

for ax, (x_column, title) in zip(
    axes,
    comparisons
):
    sns.scatterplot(
        data=price_plot_data,
        x=x_column,
        y=awarded_price_column,
        alpha=0.25,
        s=18,
        color='#4C78A8',
        edgecolor=None,
        ax=ax
    )

    lower_limit = min(
        price_plot_data[x_column].min(),
        price_plot_data[
            awarded_price_column
        ].min()
    )

    upper_limit = max(
        price_plot_data[x_column].max(),
        price_plot_data[
            awarded_price_column
        ].max()
    )

    ax.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle='--',
        color='#D62728',
        linewidth=1.5,
        label='Equal price'
    )

    ax.set_xscale('log')
    ax.set_yscale('log')

    ax.set_title(title)
    ax.set_xlabel('Comparison value (THB, log scale)')
    ax.set_ylabel('Awarded price (THB, log scale)')
    ax.legend()

plt.tight_layout()

figure_path = figure_directory / 'fig03_03_awarded_price_comparison.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

### ข้อค้นพบ: ราคาเคลื่อนไหวไปในทิศทางเดียวกัน แต่มีค่าที่ต้องตรวจเพิ่ม

**พบอะไร:** ราคาที่ตกลงมีความสัมพันธ์เชิงบวกกับงบประมาณและราคากลาง
จุดส่วนใหญ่อยู่ใกล้หรือใต้เส้นราคาเท่ากัน แต่การเปรียบเทียบกับราคากลาง
มีจุดที่อยู่ห่างจากเส้นมากกว่าการเปรียบเทียบกับงบประมาณ

**หมายความว่าอย่างไร:** โครงการส่วนใหญ่มีราคาตกลงไม่สูงกว่ากรอบราคา
แต่ค่าที่ห่างมากอาจเกิดได้ทั้งจากเหตุการณ์จริงและปัญหาคุณภาพข้อมูล
โดยเฉพาะกรณีราคากลางต่ำผิดสังเกต

**คำถามถัดไป:** ต้องตรวจความสมเหตุสมผลของราคากลางก่อน แล้วจึงกำหนด
เกณฑ์ส่วนต่างทั้งจำนวนเงินและร้อยละใน Notebook ตัวชี้วัด

กราฟใช้ตัวอย่าง 10,000 โครงการเพื่อให้แสดงผลได้ชัดเจน
ส่วนสถิติและจำนวนโครงการคำนวณจากข้อมูลทั้งหมด


### 3.3 การเปรียบเทียบตามวิธีจัดซื้อจัดจ้าง

วิธีจัดซื้ออาจสัมพันธ์กับขนาดโครงการและระดับส่วนต่างราคา
ส่วนนี้จึงเปรียบเทียบจำนวนโครงการ มูลค่า และส่วนต่างราคา
ระหว่างวิธีจัดซื้อทั้ง 4 วิธี

เนื่องจากส่วนต่างร้อยละมีค่าผิดสังเกตรุนแรง การเปรียบเทียบ
ค่ากลางจะใช้ค่ามัธยฐานแทนค่าเฉลี่ย

In [ ]:
method_column = (
    'ชื่อวิธีการจัดซื้อจัดจ้าง'
)

method_analysis = (
    project_data
    .assign(
        awarded_above_budget=(
            project_data[
                awarded_price_column
            ]
            .gt(
                project_data[
                    budget_column
                ]
            )
        ),
        awarded_equal_budget=(
            project_data[
                awarded_price_column
            ]
            .eq(
                project_data[
                    budget_column
                ]
            )
        ),
        reference_available=(
            project_data[
                reference_price_column
            ]
            .notna()
        ),
        awarded_above_reference=(
            project_data[
                awarded_price_column
            ]
            .gt(
                project_data[
                    reference_price_column
                ]
            )
        ),
        awarded_equal_reference=(
            project_data[
                awarded_price_column
            ]
            .eq(
                project_data[
                    reference_price_column
                ]
            )
        )
    )
    .groupby(method_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        ),
        above_budget_count=(
            'awarded_above_budget',
            'sum'
        ),
        equal_budget_count=(
            'awarded_equal_budget',
            'sum'
        ),
        reference_available_count=(
            'reference_available',
            'sum'
        ),
        above_reference_count=(
            'awarded_above_reference',
            'sum'
        ),
        equal_reference_count=(
            'awarded_equal_reference',
            'sum'
        )
    )
    .reset_index()
)

method_analysis['project_pct'] = (
    method_analysis['project_count']
    .div(
        method_analysis[
            'project_count'
        ].sum()
    )
    .mul(100)
)

method_analysis['budget_share_pct'] = (
    method_analysis['total_budget']
    .div(
        method_analysis[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

method_analysis['above_budget_pct'] = (
    method_analysis['above_budget_count']
    .div(
        method_analysis[
            'project_count'
        ]
    )
    .mul(100)
)

method_analysis['equal_budget_pct'] = (
    method_analysis['equal_budget_count']
    .div(
        method_analysis[
            'project_count'
        ]
    )
    .mul(100)
)

method_analysis['above_reference_pct'] = (
    method_analysis['above_reference_count']
    .div(
        method_analysis[
            'reference_available_count'
        ]
    )
    .mul(100)
)

method_analysis['equal_reference_pct'] = (
    method_analysis['equal_reference_count']
    .div(
        method_analysis[
            'reference_available_count'
        ]
    )
    .mul(100)
)

method_analysis = (
    method_analysis
    .sort_values(
        'project_count',
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
display(
    method_analysis[
        [
            method_column,
            'project_count',
            'project_pct',
            'budget_share_pct',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct',
            'equal_budget_pct',
            'above_budget_pct',
            'equal_reference_pct',
            'above_reference_pct'
        ]
    ]
)

### ข้อค้นพบ: จำนวนโครงการและวงเงินให้ภาพต่างกัน

**พบอะไร:** วิธีเฉพาะเจาะจงคิดเป็น 76.35% ของจำนวนโครงการ
แต่เป็นเพียง 9.33% ของวงเงินรวม และมีงบมัธยฐาน 283,000 บาท
ขณะที่ e-bidding มี 21.22% ของโครงการ แต่ครองวงเงิน 83.34%
และมีงบมัธยฐานประมาณ 3.10 ล้านบาท

e-bidding มีส่วนต่างจากงบประมาณมัธยฐาน 8.07% และจากราคากลาง
6.60% ส่วนวิธีเฉพาะเจาะจงมีค่ามัธยฐาน 0.18% และ 0.22%
โครงการเฉพาะเจาะจง 41.76% มีราคาตกลงเท่ากับงบประมาณ

**หมายความว่าอย่างไร:** วิธีเฉพาะเจาะจงเป็นกลไกหลักของโครงการ
ขนาดเล็ก ส่วน e-bidding รองรับมูลค่าส่วนใหญ่ จึงไม่ควรเปรียบเทียบ
วิธีจัดซื้อจากจำนวนโครงการเพียงอย่างเดียว

**คำถามถัดไป:** ภายในโครงการเฉพาะเจาะจง จุดกระจุกตัวบริเวณ
500,000 บาทมีขนาดเพียงใด และโครงการเหล่านั้นเกิดซ้ำภายใต้
หน่วยงานหรือผู้รับจ้างเดียวกันหรือไม่?


In [ ]:
method_label_map = {
    'เฉพาะเจาะจง': 'Specific',
    'ประกวดราคาอิเล็กทรอนิกส์ (e-bidding)': 'e-Bidding',
    'คัดเลือก': 'Selection',
    'ตกลงราคา': 'Price Agreement'
}

method_plot = (
    method_analysis
    .loc[
        method_analysis[
            'project_count'
        ].gt(2)
    ]
    .copy()
)

method_plot['procurement_method'] = (
    method_plot[method_column]
    .map(method_label_map)
)

method_plot = (
    method_plot[
        [
            'procurement_method',
            'project_pct',
            'budget_share_pct'
        ]
    ]
    .melt(
        id_vars='procurement_method',
        var_name='measure',
        value_name='percentage'
    )
)

measure_label_map = {
    'project_pct': 'Share of projects',
    'budget_share_pct': 'Share of total budget'
}

method_plot['measure'] = (
    method_plot['measure']
    .map(measure_label_map)
)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

sns.barplot(
    data=method_plot,
    y='procurement_method',
    x='percentage',
    hue='measure',
    palette=[
        '#4C78A8',
        '#E67E22'
    ],
    ax=ax
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt='%.1f%%',
        padding=3
    )

ax.set_title(
    'Project Share and Budget Share '
    'by Procurement Method'
)

ax.set_xlabel('Share (%)')
ax.set_ylabel('Procurement method')
ax.legend(title='')

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_04_project_and_budget_share_by_method.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

In [ ]:
budget_concentration = (
    project_data
    .assign(
        budget_le_500k=(
            project_data[
                budget_column
            ].le(500_000)
        ),
        budget_400k_to_500k=(
            project_data[
                budget_column
            ].gt(400_000)
            & project_data[
                budget_column
            ].le(500_000)
        ),
        budget_490k_to_500k=(
            project_data[
                budget_column
            ].ge(490_000)
            & project_data[
                budget_column
            ].le(500_000)
        ),
        budget_exactly_500k=(
            project_data[
                budget_column
            ].eq(500_000)
        )
    )
    .groupby(method_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        budget_le_500k_count=(
            'budget_le_500k',
            'sum'
        ),
        budget_400k_to_500k_count=(
            'budget_400k_to_500k',
            'sum'
        ),
        budget_490k_to_500k_count=(
            'budget_490k_to_500k',
            'sum'
        ),
        budget_exactly_500k_count=(
            'budget_exactly_500k',
            'sum'
        )
    )
    .reset_index()
)

count_columns = [
    'budget_le_500k_count',
    'budget_400k_to_500k_count',
    'budget_490k_to_500k_count',
    'budget_exactly_500k_count'
]

for column in count_columns:
    percentage_column = (
        column
        .replace('_count', '_pct')
    )

    budget_concentration[
        percentage_column
    ] = (
        budget_concentration[column]
        .div(
            budget_concentration[
                'project_count'
            ]
        )
        .mul(100)
    )

display(
    budget_concentration[
        [
            method_column,
            'project_count',
            'budget_le_500k_pct',
            'budget_400k_to_500k_pct',
            'budget_490k_to_500k_pct',
            'budget_exactly_500k_pct'
        ]
    ]
)

### การกระจุกตัวของวงเงินตามวิธีจัดซื้อจัดจ้าง

การกระจุกตัวของวงเงินบริเวณ 500,000 บาทเกิดขึ้นเด่นชัด
ในโครงการที่ใช้วิธีเฉพาะเจาะจง

โครงการเฉพาะเจาะจงจำนวน 99.57% มีวงเงินไม่เกิน 500,000 บาท
โดย 32.72% อยู่ในช่วงมากกว่า 400,000 ถึง 500,000 บาท
และ 19.14% อยู่ในช่วง 490,000–500,000 บาท นอกจากนี้
4.75% มีวงเงินเท่ากับ 500,000 บาทพอดี

ในทางตรงกันข้าม โครงการที่มีวงเงินไม่เกิน 500,000 บาท
พบเพียง 2.25% ในวิธีคัดเลือก และ 1.19% ใน e-bidding

ผลดังกล่าวแสดงว่าวิธีจัดซื้อมีความสัมพันธ์อย่างมากกับช่วงวงเงิน
และการกระจุกตัวใกล้ 500,000 บาทควรถูกนำไปวิเคราะห์เพิ่มเติม
อย่างไรก็ตาม ข้อมูลนี้ยังไม่เพียงพอที่จะสรุปว่าเกิดการแบ่งโครงการ
หรือการหลีกเลี่ยงกระบวนการจัดซื้อ

### 3.4 การวิเคราะห์ตามหน่วยงาน

ส่วนนี้เปรียบเทียบหน่วยงานในสองมิติ ได้แก่ จำนวนโครงการ
และวงเงินงบประมาณรวม เพื่อแยกหน่วยงานที่ดำเนินโครงการจำนวนมาก
ออกจากหน่วยงานที่รับผิดชอบโครงการมูลค่าสูง

การวิเคราะห์ใช้ `ชื่อหน่วยงาน` จากข้อมูลระดับโครงการ
และยังไม่รวม `ชื่อหน่วยงานย่อย`

In [ ]:
agency_column = 'ชื่อหน่วยงาน'

agency_summary = (
    project_data
    .groupby(agency_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        total_awarded_price=(
            awarded_price_column,
            'sum'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        )
    )
    .reset_index()
)

agency_summary['project_share_pct'] = (
    agency_summary['project_count']
    .div(
        agency_summary[
            'project_count'
        ].sum()
    )
    .mul(100)
)

agency_summary['budget_share_pct'] = (
    agency_summary['total_budget']
    .div(
        agency_summary[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

agency_summary['total_budget_billion'] = (
    agency_summary['total_budget']
    .div(1_000_000_000)
)

In [ ]:
top_agencies_by_count = (
    agency_summary
    .nlargest(
        10,
        'project_count'
    )
    [
        [
            agency_column,
            'project_count',
            'project_share_pct',
            'total_budget_billion',
            'median_budget'
        ]
    ]
)

display(top_agencies_by_count)

In [ ]:
top_agencies_by_budget = (
    agency_summary
    .nlargest(
        10,
        'total_budget'
    )
    [
        [
            agency_column,
            'total_budget_billion',
            'budget_share_pct',
            'project_count',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct'
        ]
    ]
)

display(top_agencies_by_budget)

### ข้อค้นพบตามหน่วยงาน

กรมทางหลวงมีจำนวนโครงการมากที่สุด 5,774 โครงการ และมีวงเงินรวม
สูงที่สุดประมาณ 95.33 พันล้านบาท หรือ 20.03% ของวงเงินทั้งหมด
จึงเป็นหน่วยงานที่มีบทบาทสูงทั้งด้านจำนวนและมูลค่าโครงการ

กรมทางหลวงชนบทและกรมชลประทานอยู่ในอันดับสูงทั้งสองมิติเช่นกัน
ขณะที่กรมโยธาธิการและผังเมืองมีเพียง 780 โครงการ แต่มีวงเงินรวม
สูงถึง 32.31 พันล้านบาท และมีวงเงินมัธยฐาน 20 ล้านบาท
สะท้อนว่าเป็นหน่วยงานที่มีโครงการขนาดใหญ่โดยทั่วไป

ในทางตรงกันข้าม กรมการปกครองและกรมพัฒนาที่ดินมีจำนวนโครงการ
มากเป็นอันดับต้น แต่มีวงเงินมัธยฐานเพียง 373,000 และ 189,600 บาท
ตามลำดับ แสดงให้เห็นว่าการจัดอันดับจากจำนวนโครงการและวงเงินรวม
ให้ภาพที่แตกต่างกัน

หน่วยงาน 4 อันดับแรกตามวงเงินรวมครองวงเงินประมาณ 44.25%
ของข้อมูลทั้งหมด โดยส่วนใหญ่เป็นหน่วยงานที่รับผิดชอบ
โครงสร้างพื้นฐานด้านถนน น้ำ และงานโยธา

In [ ]:
agency_label_map = {
    'กรมทางหลวง': 'Department of Highways',
    'กรมทางหลวงชนบท': 'Department of Rural Roads',
    'กรมชลประทาน': 'Royal Irrigation Department',
    'กรมโยธาธิการและผังเมือง': (
        'Department of Public Works'
    ),
    'การประปาส่วนภูมิภาค': (
        'Provincial Waterworks Authority'
    ),
    'กรุงเทพมหานคร': (
        'Bangkok Metropolitan Administration'
    ),
    'การไฟฟ้านครหลวง': (
        'Metropolitan Electricity Authority'
    ),
    'สำนักงานตำรวจแห่งชาติ': (
        'Royal Thai Police'
    ),
    'กรมทรัพยากรน้ำ': (
        'Department of Water Resources'
    ),
    'กรมทรัพยากรน้ำบาดาล': (
        'Department of Groundwater Resources'
    )
}

agency_budget_plot = (
    agency_summary
    .nlargest(
        10,
        'total_budget'
    )
    .sort_values(
        'total_budget_billion',
        ascending=True
    )
    .copy()
)

agency_budget_plot['agency_label'] = (
    agency_budget_plot[agency_column]
    .map(agency_label_map)
)

fig, ax = plt.subplots(
    figsize=(11, 7)
)

bars = ax.barh(
    agency_budget_plot['agency_label'],
    agency_budget_plot[
        'total_budget_billion'
    ],
    color='#4C78A8'
)

ax.bar_label(
    bars,
    labels=[
        f'{budget:.1f}B'
        for budget
        in agency_budget_plot[
            'total_budget_billion'
        ]
    ],
    padding=3
)

ax.set_title(
    'Top 10 Government Agencies '
    'by Construction Budget'
)

ax.set_xlabel('Total budget (billion THB)')
ax.set_ylabel('Government agency')

ax.spines[
    ['top', 'right', 'left']
].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_05_top_agencies_by_construction_budget.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

### 3.5 การกระจายตามพื้นที่

ส่วนนี้เปรียบเทียบจำนวนโครงการและวงเงินรวมในแต่ละจังหวัด
เพื่อดูว่าพื้นที่ที่มีโครงการจำนวนมากเป็นพื้นที่เดียวกับ
พื้นที่ที่ได้รับวงเงินรวมสูงหรือไม่

ข้อมูลจังหวัดระบุสถานที่ของโครงการตามชุดข้อมูลต้นทาง
จึงไม่ได้หมายถึงที่ตั้งสำนักงานใหญ่ของหน่วยงานหรือผู้รับจ้าง

In [ ]:
province_column = 'จังหวัด'

province_summary = (
    project_data
    .groupby(province_column)
    .agg(
        project_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_budget=(
            budget_column,
            'sum'
        ),
        median_budget=(
            budget_column,
            'median'
        ),
        total_awarded_price=(
            awarded_price_column,
            'sum'
        ),
        median_budget_saving_pct=(
            'budget_saving_pct',
            'median'
        ),
        median_reference_discount_pct=(
            'reference_discount_pct',
            'median'
        )
    )
    .reset_index()
)

province_summary['project_share_pct'] = (
    province_summary['project_count']
    .div(
        province_summary[
            'project_count'
        ].sum()
    )
    .mul(100)
)

province_summary['budget_share_pct'] = (
    province_summary['total_budget']
    .div(
        province_summary[
            'total_budget'
        ].sum()
    )
    .mul(100)
)

province_summary['total_budget_billion'] = (
    province_summary['total_budget']
    .div(1_000_000_000)
)

In [ ]:
display(
    province_summary
    .nlargest(
        10,
        'project_count'
    )
    [
        [
            province_column,
            'project_count',
            'project_share_pct',
            'total_budget_billion',
            'median_budget'
        ]
    ]
)

In [ ]:
display(
    province_summary
    .nlargest(
        10,
        'total_budget'
    )
    [
        [
            province_column,
            'total_budget_billion',
            'budget_share_pct',
            'project_count',
            'median_budget',
            'median_budget_saving_pct',
            'median_reference_discount_pct'
        ]
    ]
)

### ข้อค้นพบตามพื้นที่

นครราชสีมามีจำนวนโครงการมากที่สุด 8,641 โครงการ
รองลงมาคืออุบลราชธานี 7,181 โครงการ และขอนแก่น
6,605 โครงการ โดยจังหวัดที่มีโครงการจำนวนมากส่วนใหญ่
มีวงเงินมัธยฐานประมาณ 300,000–400,000 บาท

กรุงเทพมหานครมีจำนวน 4,825 โครงการ หรือ 2.70%
ของโครงการทั้งหมด แต่มีวงเงินรวมสูงถึง 177.54 พันล้านบาท
คิดเป็น 37.30% ของวงเงินทั้งหมด และมีวงเงินมัธยฐาน
4.46 ล้านบาท

ผลนี้แสดงว่าจังหวัดที่มีจำนวนโครงการมากที่สุดไม่จำเป็นต้องเป็น
จังหวัดที่มีวงเงินรวมสูงที่สุด โดยกรุงเทพมหานครมีโครงการ
ขนาดใหญ่กว่าจังหวัดอื่นโดยทั่วไป

การวิเคราะห์พื้นที่จะใช้เป็นบริบทประกอบการอธิบายข้อมูล
แต่จะไม่เป็นแกนหลักของตัวชี้วัดเพื่อการตรวจสอบ

### 3.6 การทำความเข้าใจข้อมูลผู้รับจ้าง

การวิเคราะห์ผู้รับจ้างใช้ข้อมูลทุกแถวจาก `construction_data`
เพราะหนึ่งโครงการอาจมีหลายสัญญา หลายผู้ชนะ หรือมีโครงสร้าง
กิจการร่วมค้า

ก่อนสรุปจำนวนโครงการและมูลค่าตามผู้รับจ้าง จะตรวจสอบ
ความสัมพันธ์ระหว่างเลขประจำตัวนิติบุคคลกับชื่อผู้ชนะ
และตรวจสอบว่าการรวมวงเงินทุกแถวทำให้เกิดการนับซ้ำจาก
แถวสมาชิกกิจการร่วมค้าหรือไม่

In [ ]:
supplier_id_column = (
    'เลขประจำตัวนิติบุคคล 13 หลัก'
)

supplier_name_column = (
    'ชื่อผู้ชนะการเสนอราคา'
)

print('Most frequent supplier IDs:')

display(
    construction_data[
        supplier_id_column
    ]
    .value_counts(
        dropna=False
    )
    .head(15)
)

print('\nMost frequent supplier names:')

display(
    construction_data[
        supplier_name_column
    ]
    .value_counts(
        dropna=False
    )
    .head(15)
)

In [ ]:
supplier_id_profile = (
    construction_data
    .groupby(supplier_id_column)
    [supplier_name_column]
    .nunique()
)

supplier_name_profile = (
    construction_data
    .groupby(supplier_name_column)
    [supplier_id_column]
    .nunique()
)

supplier_relationship_summary = pd.Series({
    'Unique supplier IDs': (
        construction_data[
            supplier_id_column
        ].nunique()
    ),
    'Unique supplier names': (
        construction_data[
            supplier_name_column
        ].nunique()
    ),
    'Supplier IDs linked to multiple names': (
        supplier_id_profile.gt(1).sum()
    ),
    'Supplier names linked to multiple IDs': (
        supplier_name_profile.gt(1).sum()
    )
})

display(
    supplier_relationship_summary
    .to_frame(name='value')
)

### ข้อค้นพบเกี่ยวกับรหัสผู้รับจ้าง

ข้อมูลมีเลขประจำตัวนิติบุคคลที่ไม่ซ้ำ 33,685 ค่า
และชื่อผู้ชนะการเสนอราคาที่ไม่ซ้ำ 38,826 ค่า

พบเลขนิติบุคคล 4,234 รหัสที่เชื่อมโยงกับชื่อผู้ชนะมากกว่าหนึ่งรูปแบบ
ซึ่งอาจเกิดจากความแตกต่างของการสะกด คำนำหน้า สาขา
หรือรูปแบบการบันทึกชื่อ ขณะเดียวกันพบชื่อผู้ชนะ 253 ชื่อ
ที่เชื่อมโยงกับเลขนิติบุคคลมากกว่าหนึ่งรหัส

ดังนั้น การวิเคราะห์จะใช้เลขประจำตัวนิติบุคคลเป็นรหัสหลัก
และใช้ชื่อที่พบบ่อยที่สุดของแต่ละรหัสเป็นชื่อสำหรับแสดงผล
โดยไม่รวมผู้รับจ้างจากชื่อเพียงอย่างเดียว

In [ ]:
contract_budget_column = 'วงเงินงบประมาณในสัญญา (บาท)'

project_awarded_total = (
    project_data[
        awarded_price_column
    ].sum()
)

raw_contract_budget_total = (
    construction_data[
        contract_budget_column
    ].sum()
)

raw_value_difference = (
    raw_contract_budget_total
    - project_awarded_total
)

raw_value_reconciliation = pd.Series({
    'Project awarded total': (
        project_awarded_total
    ),
    'Raw contract budget total': (
        raw_contract_budget_total
    ),
    'Raw difference': (
        raw_value_difference
    ),
    'Raw difference pct': (
        raw_value_difference
        / project_awarded_total
        * 100
    ),
    'Missing contract budget rows': (
        construction_data[
            contract_budget_column
        ]
        .isna()
        .sum()
    )
})

display(
    raw_value_reconciliation
    .to_frame(name='value')
)

In [ ]:
raw_contract_value_by_project = (
    construction_data
    .groupby('รหัสโครงการ')[
        contract_budget_column
    ]
    .sum(min_count=1)
    .rename('raw_contract_budget_sum')
    .reset_index()
)

raw_project_value_check = (
    project_data[
        [
            'รหัสโครงการ',
            awarded_price_column
        ]
    ]
    .merge(
        raw_contract_value_by_project,
        on='รหัสโครงการ',
        how='left'
    )
)

raw_project_value_check['difference'] = (
    raw_project_value_check[
        'raw_contract_budget_sum'
    ]
    - raw_project_value_check[
        awarded_price_column
    ]
)

raw_project_value_check['is_matched'] = np.isclose(
    raw_project_value_check[
        'raw_contract_budget_sum'
    ],
    raw_project_value_check[
        awarded_price_column
    ],
    rtol=0,
    atol=1,
    equal_nan=False
)

raw_reconciliation_summary = pd.Series({
    'Projects checked': (
        len(raw_project_value_check)
    ),
    'Matched projects': (
        raw_project_value_check[
            'is_matched'
        ].sum()
    ),
    'Unmatched projects': (
        ~raw_project_value_check[
            'is_matched'
        ]
    ).sum(),
    'Matched pct': (
        raw_project_value_check[
            'is_matched'
        ].mean()
        * 100
    )
})

display(
    raw_reconciliation_summary
    .to_frame(name='value')
)

### ตรวจพบโครงสร้างกิจการร่วมค้าจากผลรวมระดับสัญญา

เมื่อรวมวงเงินทุกแถวโดยตรง ยอดระดับสัญญาตรงกับราคาที่ตกลง
178,616 จาก 178,978 โครงการ หรือ 99.80% และพบส่วนต่าง
362 โครงการ ยอดรวมดิบสูงกว่าระดับโครงการประมาณ 13.23
พันล้านบาท

อย่างไรก็ตาม ผลนี้ยังไม่ถือเป็นปัญหาคุณภาพข้อมูล เพราะข้อมูล
กิจการร่วมค้าบันทึกทั้งแถวของ Consortium ซึ่งมีวงเงินเต็มสัญญา
และแถวของสมาชิกซึ่งมีวงเงินตามส่วนแบ่ง หากรวมทั้งหมดพร้อมกัน
จึงเกิดการนับมูลค่าซ้ำ

สำหรับการวิเคราะห์ Supplier Entity จะเก็บแถว Consortium
และตัดเฉพาะแถวที่ชื่อผู้ชนะระบุว่า `สัญญากิจการค้าร่วม`
ก่อนตรวจสอบยอดอีกครั้ง

In [ ]:
construction_data[
    'is_joint_venture_member'
] = (
    construction_data[
        supplier_name_column
    ]
    .astype('string')
    .str.contains(
        'สัญญากิจการค้าร่วม',
        na=False
    )
)

joint_venture_member_summary = pd.Series({
    'Joint-venture member rows': (
        construction_data[
            'is_joint_venture_member'
        ].sum()
    ),
    'Projects with member rows': (
        construction_data.loc[
            construction_data[
                'is_joint_venture_member'
            ],
            'รหัสโครงการ'
        ].nunique()
    ),
    'Member-row contract value': (
        construction_data.loc[
            construction_data[
                'is_joint_venture_member'
            ],
            contract_budget_column
        ].sum()
    )
})

display(
    joint_venture_member_summary
    .to_frame(name='value')
)

In [ ]:
supplier_entity_data = (
    construction_data
    .loc[
        ~construction_data[
            'is_joint_venture_member'
        ]
    ]
    .copy()
)

entity_value_by_project = (
    supplier_entity_data
    .groupby('รหัสโครงการ')[
        contract_budget_column
    ]
    .sum(min_count=1)
    .rename('entity_contract_value_sum')
    .reset_index()
)

entity_reconciliation = (
    project_data[
        [
            'รหัสโครงการ',
            awarded_price_column
        ]
    ]
    .merge(
        entity_value_by_project,
        on='รหัสโครงการ',
        how='left'
    )
)

entity_reconciliation['difference'] = (
    entity_reconciliation[
        'entity_contract_value_sum'
    ]
    - entity_reconciliation[
        awarded_price_column
    ]
)

entity_reconciliation['is_matched'] = np.isclose(
    entity_reconciliation[
        'entity_contract_value_sum'
    ],
    entity_reconciliation[
        awarded_price_column
    ],
    rtol=0,
    atol=1,
    equal_nan=False
)

entity_reconciliation_summary = pd.Series({
    'Supplier entity rows': (
        len(supplier_entity_data)
    ),
    'Projects checked': (
        len(entity_reconciliation)
    ),
    'Matched projects': (
        entity_reconciliation[
            'is_matched'
        ].sum()
    ),
    'Unmatched projects': (
        ~entity_reconciliation[
            'is_matched'
        ]
    ).sum(),
    'Matched pct': (
        entity_reconciliation[
            'is_matched'
        ].mean()
        * 100
    ),
    'Total absolute difference': (
        entity_reconciliation[
            'difference'
        ].abs().sum()
    )
})

display(
    entity_reconciliation_summary
    .to_frame(name='value')
)

In [ ]:
matched_entity_project_ids = set(
    entity_reconciliation.loc[
        entity_reconciliation[
            'is_matched'
        ],
        'รหัสโครงการ'
    ]
)

supplier_analysis_data = (
    supplier_entity_data
    .loc[
        supplier_entity_data[
            'รหัสโครงการ'
        ].isin(
            matched_entity_project_ids
        )
        & supplier_entity_data[
            contract_budget_column
        ].notna()
    ]
    .copy()
)

print(
    f'Supplier entity rows used: '
    f'{len(supplier_analysis_data):,}'
)

print(
    f'Reconciled projects used: '
    f'{supplier_analysis_data["รหัสโครงการ"].nunique():,}'
)

In [ ]:
supplier_name_lookup = (
    supplier_analysis_data
    .groupby(
        [
            supplier_id_column,
            supplier_name_column
        ]
    )
    .size()
    .rename('name_count')
    .reset_index()
    .sort_values(
        [
            supplier_id_column,
            'name_count',
            supplier_name_column
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .drop_duplicates(
        subset=supplier_id_column,
        keep='first'
    )[
        [
            supplier_id_column,
            supplier_name_column
        ]
    ]
    .rename(
        columns={
            supplier_name_column:
            'supplier_display_name'
        }
    )
)

supplier_summary = (
    supplier_analysis_data
    .groupby(supplier_id_column)
    .agg(
        supplier_project_count=(
            'รหัสโครงการ',
            'nunique'
        ),
        supplier_record_count=(
            'รหัสโครงการ',
            'size'
        ),
        total_contract_value=(
            contract_budget_column,
            'sum'
        ),
        median_contract_value=(
            contract_budget_column,
            'median'
        )
    )
    .reset_index()
    .merge(
        supplier_name_lookup,
        on=supplier_id_column,
        how='left'
    )
)

supplier_summary[
    'total_contract_value_million'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .div(1_000_000)
)

supplier_summary[
    'value_share_pct'
] = (
    supplier_summary[
        'total_contract_value'
    ]
    .div(
        supplier_summary[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

In [ ]:
top_suppliers_by_projects = (
    supplier_summary
    .nlargest(
        15,
        'supplier_project_count'
    )[
        [
            supplier_id_column,
            'supplier_display_name',
            'supplier_project_count',
            'supplier_record_count',
            'total_contract_value_million'
        ]
    ]
)

top_suppliers_by_value = (
    supplier_summary
    .nlargest(
        15,
        'total_contract_value'
    )[
        [
            supplier_id_column,
            'supplier_display_name',
            'total_contract_value_million',
            'value_share_pct',
            'supplier_project_count',
            'median_contract_value'
        ]
    ]
)

display(top_suppliers_by_projects)
display(top_suppliers_by_value)

### ข้อค้นพบเกี่ยวกับผู้รับจ้าง

พบแถวสมาชิกกิจการร่วมค้า 710 แถว ครอบคลุม 362 โครงการ
และมีมูลค่ารวมประมาณ 13.06 พันล้านบาท

เมื่อตัดแถวสมาชิกออกและเก็บ Consortium เป็น Supplier Entity
ผลการตรวจสอบเพิ่มเป็น 178,967 จาก 178,978 โครงการ
หรือ 99.994% เหลือโครงการที่ยอดยังไม่ตรงกัน 11 โครงการ

ผู้รับจ้างที่ได้รับโครงการจำนวนมากที่สุดไม่ใช่ผู้รับจ้าง
ที่ได้รับมูลค่ารวมสูงที่สุด สะท้อนรูปแบบสองกลุ่ม ได้แก่
ผู้รับจ้างที่ได้โครงการขนาดเล็กจำนวนมาก และผู้รับจ้างที่ได้
โครงการมูลค่าสูงเพียงไม่กี่รายการ

การคำนวณมูลค่าผู้รับจ้างต่อจากนี้ใช้เฉพาะ Supplier Entity
ในโครงการที่ผ่านการตรวจสอบยอดหลังปรับกิจการร่วมค้าแล้ว

In [ ]:
supplier_pareto = (
    supplier_summary
    .sort_values(
        'total_contract_value',
        ascending=False
    )
    .reset_index(drop=True)
    .copy()
)

supplier_pareto['supplier_rank'] = (
    supplier_pareto.index + 1
)

supplier_pareto['cumulative_value_pct'] = (
    supplier_pareto[
        'total_contract_value'
    ]
    .cumsum()
    .div(
        supplier_pareto[
            'total_contract_value'
        ].sum()
    )
    .mul(100)
)

total_suppliers = len(
    supplier_pareto
)

supplier_count_50 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(50)
    .idxmax()
    + 1
)

supplier_count_80 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(80)
    .idxmax()
    + 1
)

supplier_count_90 = (
    supplier_pareto[
        'cumulative_value_pct'
    ]
    .ge(90)
    .idxmax()
    + 1
)

In [ ]:
supplier_concentration_summary = pd.Series({
    'Supplier entities': (
        total_suppliers
    ),
    'Top 1 value share (%)': (
        supplier_pareto
        .head(1)[
            'value_share_pct'
        ].sum()
    ),
    'Top 10 value share (%)': (
        supplier_pareto
        .head(10)[
            'value_share_pct'
        ].sum()
    ),
    'Top 100 value share (%)': (
        supplier_pareto
        .head(100)[
            'value_share_pct'
        ].sum()
    ),
    'Suppliers accounting for 50%': (
        supplier_count_50
    ),
    'Supplier pct accounting for 50%': (
        supplier_count_50
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 80%': (
        supplier_count_80
    ),
    'Supplier pct accounting for 80%': (
        supplier_count_80
        / total_suppliers
        * 100
    ),
    'Suppliers accounting for 90%': (
        supplier_count_90
    ),
    'Supplier pct accounting for 90%': (
        supplier_count_90
        / total_suppliers
        * 100
    )
})

display(
    supplier_concentration_summary
    .to_frame(name='value')
)

### ข้อค้นพบ: มูลค่ากระจุกในผู้รับจ้างส่วนน้อย แต่ไม่มีรายเดียวครองภาพรวม

หลังจัดกิจการร่วมค้าเป็น Supplier Entity อย่างเหมาะสม
พบผู้รับจ้าง 33,570 ราย ผู้รับจ้างรายใหญ่ที่สุดครองมูลค่า 1.51%
10 อันดับแรกครอง 7.42% และ 100 อันดับแรกครอง 27.83%

กราฟ Pareto แสดงสัดส่วนผู้รับจ้างสะสมเทียบกับมูลค่าสัญญาสะสม
ไม่ใช่หลักฐานการผูกขาด พบว่าผู้รับจ้าง 1,870 ราย หรือ 5.57%
ครองมูลค่า 80% ของทั้งหมด จึงมีการกระจุกตัวเชิงมูลค่า
แต่ไม่ได้ถูกครองโดยผู้รับจ้างรายเดียวในระดับประเทศ

คำถามที่เหมาะสมกว่าคือ ภายในหน่วยงานย่อยแต่ละแห่ง
มีผู้รับจ้างรายใดได้รับทั้งจำนวนโครงการและมูลค่าในสัดส่วนสูงหรือไม่
คำถามนี้จะถูกพัฒนาเป็นตัวชี้วัดใน Notebook 04


In [ ]:
supplier_pareto[
    'cumulative_supplier_pct'
] = (
    supplier_pareto[
        'supplier_rank'
    ]
    .div(total_suppliers)
    .mul(100)
)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    supplier_pareto[
        'cumulative_supplier_pct'
    ],
    supplier_pareto[
        'cumulative_value_pct'
    ],
    color='#4C78A8',
    linewidth=2
)

ax.axhline(
    y=80,
    color='#D62728',
    linestyle='--',
    linewidth=1.5,
    label='80% of contract value'
)

ax.axvline(
    x=(
        supplier_count_80
        / total_suppliers
        * 100
    ),
    color='#E67E22',
    linestyle='--',
    linewidth=1.5,
    label=(
        f'{supplier_count_80:,} suppliers '
        f'({supplier_count_80 / total_suppliers * 100:.1f}%)'
    )
)

ax.scatter(
    supplier_count_80
    / total_suppliers
    * 100,
    80,
    color='#D62728',
    s=60,
    zorder=3
)

ax.set_title(
    'Cumulative Contract Value '
    'by Supplier Share'
)

ax.set_xlabel(
    'Cumulative share of suppliers (%)'
)

ax.set_ylabel(
    'Cumulative share of contract value (%)'
)

ax.set_xlim(0, 100)
ax.set_ylim(0, 100)

ax.legend(
    loc='lower right'
)

ax.spines[
    ['top', 'right']
].set_visible(False)

plt.tight_layout()

figure_path = figure_directory / 'fig03_06_cumulative_contract_value_by_supplier.png'
fig.savefig(
    figure_path,
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print(f'Figure saved: {figure_path}')

## 4. สรุปผล EDA และประเด็นสำหรับวิเคราะห์ต่อ

### 4.1 การกระจายของมูลค่าโครงการ

โครงการจ้างก่อสร้างมีวงเงินงบประมาณแบบเบ้ขวาอย่างมาก
โดย 76.34% ของโครงการมีวงเงินไม่เกิน 500,000 บาท
ขณะที่โครงการมูลค่าเกิน 50 ล้านบาทมีเพียง 0.49%

แม้โครงการขนาดใหญ่มีจำนวนน้อย แต่มีอิทธิพลต่อมูลค่ารวมสูง
จึงไม่ควรประเมินความสำคัญจากจำนวนโครงการเพียงอย่างเดียว

### 4.2 การกระจุกตัวบริเวณ 500,000 บาท

โครงการเฉพาะเจาะจง 99.57% มีวงเงินไม่เกิน 500,000 บาท
และ 19.14% อยู่ในช่วง 490,000–500,000 บาท

รูปแบบนี้ควรตรวจสอบร่วมกับหน่วยงานย่อย ผู้รับจ้าง
ชื่อโครงการ และวันที่เกิดรายการ แต่ยังไม่สามารถสรุปว่า
เกิดการแบ่งโครงการหรือหลีกเลี่ยงขั้นตอนจัดซื้อ

### 4.3 ความแตกต่างตามวิธีจัดซื้อ

วิธีเฉพาะเจาะจงคิดเป็น 76.35% ของจำนวนโครงการ
แต่มีวงเงินเพียง 9.33% ของทั้งหมด ส่วน e-bidding
มีจำนวน 21.22% แต่ครองวงเงิน 83.34%

e-bidding มีส่วนต่างจากราคากลางมัธยฐาน 6.60%
สูงกว่าวิธีเฉพาะเจาะจงซึ่งมีค่ามัธยฐาน 0.22%

### 4.4 ความแตกต่างตามหน่วยงานและพื้นที่

กรมทางหลวงมีวงเงินรวมสูงที่สุดประมาณ 95.33 พันล้านบาท
หรือ 20.03% ของวงเงินทั้งหมด

กรุงเทพมหานครมีเพียง 2.70% ของจำนวนโครงการ
แต่ครองวงเงินรวม 37.30% แสดงว่าจำนวนโครงการและมูลค่า
ให้ภาพการกระจายที่แตกต่างกัน ข้อมูลพื้นที่จึงใช้เป็นบริบท
ประกอบและไม่ใช่แกนหลักของตัวชี้วัด

### 4.5 การกระจุกตัวของผู้รับจ้าง

หลังจัดการโครงสร้างกิจการร่วมค้า พบ Supplier Entity 33,570 ราย
โดยผู้รับจ้าง 1,870 ราย หรือ 5.57% ครองมูลค่าสัญญา 80%
ขณะที่ 10 อันดับแรกครองมูลค่ารวมเพียง 7.42%

ผลนี้สนับสนุนให้วิเคราะห์การกระจุกตัวภายในหน่วยงานย่อย
มากกว่าพิจารณาอันดับผู้รับจ้างในภาพรวมประเทศเพียงอย่างเดียว

### 4.6 ประเด็นคุณภาพข้อมูล

การตรวจยอดดิบพบส่วนต่าง 362 โครงการ แต่ส่วนใหญ่เกิดจาก
การบันทึกทั้ง Consortium และสมาชิกกิจการร่วมค้า หลังตัดเฉพาะ
แถวสมาชิกออก เหลือ Entity-value mismatch 11 โครงการ

ดังนั้น 362 โครงการไม่ถูกจัดเป็นปัญหาคุณภาพข้อมูลโดยอัตโนมัติ
ส่วน 11 โครงการที่ยังไม่ตรงกันจะส่งต่อเพื่อพิจารณาเป็น
Data-quality Indicator ใน Notebook ถัดไป

### 4.7 ประเด็นที่ส่งต่อไปวิเคราะห์เป็นตัวชี้วัด

1. ส่วนต่างราคาเหนือวงเงินงบประมาณหรือราคากลางอย่างมีสาระสำคัญ
2. โครงการเฉพาะเจาะจงใกล้ 500,000 บาทที่เกิดซ้ำภายใต้
   หน่วยงานย่อย ผู้รับจ้าง และวันที่เดียวกัน
3. การกระจุกตัวสูงของผู้รับจ้างภายในหน่วยงานย่อย
4. ราคากลางที่หายไปหรือมีสัดส่วนผิดสังเกต
5. Entity-value mismatch ที่เหลือหลังจัดการกิจการร่วมค้า

ประเด็นเหล่านี้เป็นหัวข้อสำหรับตรวจสอบต่อ
ยังไม่ใช่เกณฑ์ยืนยันการทุจริต

In [ ]:
project_output_path = (
    data_path.parent
    / 'construction_projects_2569.csv'
)

project_data.to_csv(
    project_output_path,
    index=False,
    encoding='utf-8-sig'
)

print(
    f'Project-level data saved to:\n'
    f'{project_output_path}'
)

In [ ]:
supplier_output_path = (
    data_path.parent
    / 'construction_supplier_summary_2569.csv'
)

supplier_summary.to_csv(
    supplier_output_path,
    index=False,
    encoding='utf-8-sig'
)

print(
    f'Supplier summary saved to:\n'
    f'{supplier_output_path}'
)

In [ ]:
output_files = [
    project_output_path,
    supplier_output_path
]

for file_path in output_files:
    print(
        f'{file_path.name}: '
        f'{file_path.stat().st_size / (1024 ** 2):,.2f} MB'
    )

## 5. ข้อมูลผลลัพธ์สำหรับ Notebook ตัวชี้วัด

Notebook นี้สร้างข้อมูลสำหรับใช้ในขั้นต่อไปจำนวน 2 ไฟล์:

1. `construction_projects_2569.csv`  
   ข้อมูลระดับโครงการ พร้อมตัวแปรส่วนต่างราคาและช่วงวงเงิน

2. `construction_supplier_summary_2569.csv`  
   ข้อมูลสรุประดับ Supplier Entity หลังตัดแถวสมาชิกกิจการร่วมค้า
   และใช้เลขประจำตัวนิติบุคคลเป็นรหัสหลัก

`04_construction_review_indicators_2569.ipynb` จะใช้ผลจาก EDA
กำหนดตัวชี้วัด โดยแยก Procurement-review Indicator
ออกจาก Data-quality Indicator อย่างชัดเจน